# 02 · E2 — Intervención preliminar sobre el decodificador de SAM 2.1 (Sen1Floods11)

**NO EJECUTADO. Solo E2.** Compara SAM 2.1 tiny con el *mask decoder* **congelado** (época 0) frente al **ajustado** (solo ese bloque; encoder de imagen y de prompts congelados), con pérdida BCE+Dice sobre un subconjunto pequeño y explícito de entrenamiento y evaluación en un subconjunto de validación.

* **Referencia asistida por etiquetas, no desplegable:** los puntos (1 positivo + 1 negativo por componente de agua) se simulan a partir de las etiquetas. No es el flujo automático DINOv3 → SAM y **DINOv3 no se usa**.
* **Preliminar:** semilla única (42), subconjuntos pequeños, sin test ni Bolivia. Nada aquí generaliza a otros eventos.
* La hipótesis se guarda en Drive **antes** de entrenar (celda de `prepare`). Los resultados reales aparecerán solo tras ejecutar en tu Colab.
* La línea base (etapa 3) y el teardown (etapa 4) ya existen en el repositorio; E2 los reutiliza (mismo evaluador, mismos prompts, mismo modelo verificado por hash).

## 1. Preparar el entorno (ZIP verificado, venv, pines)
Requiere GPU T4 y que `auto-prompt-sam2-e2.zip` esté en la raíz de *Mi unidad*.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'            # carpeta de trabajo en TU Drive
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-e2.zip')    # ZIP que subiste a la raíz de "Mi unidad"
EXPECTED_BUNDLE_SHA256 = 'd12e1ea7f4c8e1a8b79321d37110d086a360dba9b77aef43417f5d93167d842c'                                  # SHA-256 real del ZIP (calculado al generar este notebook)
REPO = Path('/content/auto-prompt-sam2-e2')
PYTHON = Path('/content/auto-prompt-venv/bin/python')

assert BUNDLE.is_file(), f'No encuentro {BUNDLE}. Sube auto-prompt-sam2-e2.zip a la raíz de Mi unidad en Drive.'
actual = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
assert actual == EXPECTED_BUNDLE_SHA256, f'ZIP distinto al esperado: {actual}'
print('ZIP verificado, SHA-256 =', actual)
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        assert (REPO / item.filename).resolve().is_relative_to(REPO.resolve()), 'Ruta insegura en el ZIP'
    archive.extractall(REPO)

child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', HF_HUB_DISABLE_TELEMETRY='1',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
LOG_DIR = Path(PERSIST_ROOT) / 'e2/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def command(args, log_name):
    """Ejecuta un proceso mostrando su salida en vivo y guardándola en Drive."""
    with (LOG_DIR / log_name).open('w', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; revisa la salida y el log {log_name}')

if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py'), '--venv', str(PYTHON.parent.parent)], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')
print('Entorno listo.')

In [ ]:
# Comprueba que Colab te dio una GPU (Entorno de ejecución > Cambiar tipo > T4 GPU).
command(['nvidia-smi'], 'nvidia_smi.log')
command([str(PYTHON), '-c', 'import torch; assert torch.cuda.is_available(), "NO HAY GPU: cambia el tipo de entorno a GPU"; print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)'], 'gpu_check.log')

## 2. Comprobar prerrequisitos de la etapa 2
Si falta algo, la celda se detiene con un mensaje claro: no se reconstruyen splits ni estadísticas aquí.

In [ ]:
# E2 NO descarga ni reconstruye datos: lee lo que dejó la etapa 2 del equipo en PERSIST_ROOT.
root = Path(PERSIST_ROOT)
required = {
    'raw/acquisition (adquisición Sen1Floods11)': root / 'raw/acquisition',
    'stage2/validation (informe de la etapa 2)': root / 'stage2/validation',
    'regimes/official/manifests': root / 'regimes/official/manifests',
    'regimes/official/preprocessing': root / 'regimes/official/preprocessing',
}
missing = [name for name, path in required.items() if not path.exists()]
for name, path in required.items():
    print(('OK       ' if path.exists() else 'FALTA    ') + name)
assert not missing, ('Faltan datos de la etapa 2 en tu Drive. NO continúes: pide a tu compañero que comparta la carpeta '
                     '"auto-prompt-sam2" (o ejecuta el notebook 02 del repositorio con su ZIP). Ver PASOS_E2.md, sección 3.')

## 3. Hipótesis y configuración (antes de entrenar)
Se muestran exactamente los valores de `configs/e2_intervention.yaml`. No se cambian después de ver resultados.

In [ ]:
print(json.dumps(json.loads((REPO / 'configs/e2_intervention.yaml').read_text()), indent=2, ensure_ascii=False))

## 4. Ejecutar E2 (`prepare`)
Hace en orden: pruebas unitarias, verificación del hash de SAM, **guardado de la hipótesis**, comprobación de paridad con el predictor de la etapa 3, época 0 (decoder congelado), entrenamiento solo del decoder (con auditoría de gradientes), evaluación de la mejor época y exportación de evidencia. Imprime el tiempo de cada época: no se estimó antes.

`prepare` crea un run nuevo. Si Colab se desconecta a medio entrenamiento: reabre, ejecuta las celdas 1–2 y cambia `MODE` a `'resume'` (mismo ZIP y mismo Drive; nunca dos escritores a la vez).

In [ ]:
MODE = 'prepare'   # 'resume' SOLO para continuar un run interrumpido
assert MODE in ('prepare', 'resume')
command([str(PYTHON), str(REPO / 'scripts/run_e2.py'), MODE, '--root', PERSIST_ROOT], MODE + '.log')

## 5. Ver resultados medidos
Todo lo mostrado sale de `report.json` y archivos del run; no hay números escritos a mano.

In [ ]:
from IPython.display import display, Image, HTML
import csv, html
EVIDENCE = Path(PERSIST_ROOT) / 'e2/evidence'
RUNS = sorted(p for p in EVIDENCE.iterdir() if p.is_dir() and (p / 'report.json').is_file())
assert RUNS, 'No hay un prepare completo. Conserva e2/notebook_logs/prepare.log y envíalo.'
RUN_DIR = RUNS[-1]
REPORT = json.loads((RUN_DIR / 'report.json').read_text())
print('run_id:', REPORT['run_id'], '| estado:', REPORT['status'])
print(json.dumps({'hipotesis_previa': REPORT['hypothesis']['statement'], 'brazos': REPORT['arms'],
                  'mejor_epoca': REPORT['best_epoch'], 'verificacion_hipotesis': REPORT['hypothesis_check'],
                  'congelamiento': {k: REPORT['freezing'][k] for k in ('frozen_blocks_unchanged', 'decoder_changed')},
                  'paridad_con_etapa3': REPORT['parity_with_stage3_predictor'],
                  'subconjuntos': {k: REPORT['subsets'][k] for k in ('val_stats', 'train_stats')}},
                 indent=2, ensure_ascii=False))
for name in ('history.csv', 'per_chip_val.csv'):
    rows = list(csv.reader((RUN_DIR / name).open(encoding='utf-8', newline='')))
    print(name)
    display(HTML('<table>' + ''.join('<tr>' + ''.join('<td>' + html.escape(v) + '</td>' for v in row) + '</tr>' for row in rows) + '</table>'))
display(Image(filename=str(RUN_DIR / 'curves.png')))
for figure in sorted((RUN_DIR / 'examples').glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure)))
print('Limitaciones declaradas:'); [print(' -', item) for item in REPORT['limitations']]

## 6. Recuperar en un runtime nuevo (`recover`)
1. Anota el ZIP de evidencia impreso arriba y **descárgalo** (celda siguiente).
2. *Entorno de ejecución > Desconectar y eliminar el entorno*. Vuelve a conectar con T4 GPU.
3. Ejecuta **solo** las celdas 1–2 (entorno y prerrequisitos) y la celda de abajo. **No repitas `prepare`.**

`recover` recarga SAM desde Drive en modo *offline*, verifica el hash del checkpoint, recarga el mejor decoder guardado y vuelve a evaluar ambos brazos; deben coincidir con lo guardado (IoU agrupado a 1e-6 y confusiones por chip idénticas). Si falla, **no cambies las tolerancias**: guarda el log.

In [ ]:
command([str(PYTHON), str(REPO / 'scripts/run_e2.py'), 'recover', '--root', PERSIST_ROOT], 'recover.log')

## 7. Descargar la evidencia
El ZIP contiene informe, tablas, curvas, ejemplos, predicciones y puntos. **No** incluye checkpoints, pesos, tokens ni imágenes de entrenamiento (la licencia de Sen1Floods11 sigue pendiente).

In [ ]:
from google.colab import files
ZIPS = sorted((Path(PERSIST_ROOT) / 'e2/evidence').glob('*.zip'))
assert ZIPS, 'No hay ZIP de evidencia'
print(ZIPS[-1], hashlib.sha256(ZIPS[-1].read_bytes()).hexdigest())
files.download(str(ZIPS[-1]))

## 8. Qué devolver y qué NO afirmar
Devuelve: **este notebook ejecutado** (Archivo > Descargar > .ipynb, con salidas y errores), el ZIP de evidencia y, si hubo errores, `e2/notebook_logs/*.log`.

E2 **no queda acreditada** por crear este cuaderno ni por correrlo: falta revisar los resultados, interpretarlos (incluyendo fallos y límites) y completar el cuaderno de decisiones con tus conclusiones. IA: Claude preparó código, pruebas y documentación; la comprensión y la defensa oral son tuyas.